# Bens declarados dos candidatos do Amazonas (Eleições 2026)

Análise exploratória e geração do CSV consumido pelo dashboard.

## A1. Carga e inspeção inicial

Os arquivos do TSE usam `;`, `latin-1`, vírgula decimal no arquivo de bens e `SQ_CANDIDATO` como texto.

In [ ]:
import io
import zipfile
from pathlib import Path
import pandas as pd
import plotly.express as px
import requests

BASE = 'https://cdn.tse.jus.br/estatistica/sead/odsele'
def baixar_am(url, sufixo='_AM.CSV'):
    r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=180)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    nome = next(n for n in z.namelist() if n.upper().endswith(sufixo))
    z.extract(nome, '.')
    return Path(nome).name

arq_bens = baixar_am(f'{BASE}/bem_candidato/bem_candidato_2026.zip')
arq_cand = baixar_am(f'{BASE}/consulta_cand/consulta_cand_2026.zip')
bens = pd.read_csv(arq_bens, sep=';', encoding='latin-1', decimal=',', dtype={'SQ_CANDIDATO': str})
cand = pd.read_csv(arq_cand, sep=';', encoding='latin-1', dtype={'SQ_CANDIDATO': str})
print('bens:', bens.shape); display(bens.head()); display(bens.dtypes)
print('candidatos:', cand.shape); display(cand.head()); display(cand.dtypes)
assert pd.api.types.is_numeric_dtype(bens['VR_BEM_CANDIDATO'])

## A2. Limpeza, qualidade e deduplicação

Cada linha de `bens` é um bem. Já `cand` pode repetir o candidato; por isso deduplicamos antes do join para não multiplicar o patrimônio.

In [ ]:
relevantes_bens = ['SQ_CANDIDATO', 'DS_TIPO_BEM_CANDIDATO', 'VR_BEM_CANDIDATO', 'SG_UF']
relevantes_cand = ['SQ_CANDIDATO', 'NM_URNA_CANDIDATO', 'DS_CARGO', 'SG_PARTIDO', 'DS_GENERO', 'DS_GRAU_INSTRUCAO']
display(bens[relevantes_bens].isna().sum()); display(cand[relevantes_cand].isna().sum())
print('UFs em bens:', bens['SG_UF'].value_counts(dropna=False).to_dict())
bens = bens[bens['SG_UF'].eq('AM')].copy()
cand = cand[relevantes_cand].drop_duplicates('SQ_CANDIDATO').copy()
print('candidatos únicos:', cand['SQ_CANDIDATO'].nunique())

## A3. EDA univariada

In [ ]:
display(bens['VR_BEM_CANDIDATO'].describe())
fig = px.histogram(bens, x='VR_BEM_CANDIDATO', nbins=40, title='Distribuição do valor declarado dos bens')
fig.show()
fig = px.bar(bens['DS_TIPO_BEM_CANDIDATO'].value_counts().head(15).reset_index(), x='count', y='DS_TIPO_BEM_CANDIDATO', orientation='h', title='Tipos de bem mais frequentes')
fig.show()
print('A média acima da mediana indica assimetria à direita e possível concentração de valores altos em poucos bens.')

## A4. Agregação, join e CSV limpo

Usamos `left` a partir de candidatos para preservar também quem não declarou bens; nesse caso o patrimônio é preenchido com zero.

In [ ]:
def tipo_principal(s):
    moda = s.mode()
    return moda.iloc[0] if not moda.empty else 'Não informado'
agg = bens.groupby('SQ_CANDIDATO', as_index=False).agg(
    patrimonio_total=('VR_BEM_CANDIDATO', 'sum'),
    qtd_bens=('VR_BEM_CANDIDATO', 'size'),
    tipo_principal=('DS_TIPO_BEM_CANDIDATO', tipo_principal),
)
saida = cand.merge(agg, on='SQ_CANDIDATO', how='left')
saida['patrimonio_total'] = saida['patrimonio_total'].fillna(0)
saida['qtd_bens'] = saida['qtd_bens'].fillna(0).astype(int)
saida['tipo_principal'] = saida['tipo_principal'].fillna('Sem bens declarados')
saida.to_csv('bens_am_por_candidato.csv', index=False, encoding='utf-8')
display(saida.sort_values('patrimonio_total', ascending=False).head())

## A5. Comparações por grupos

In [ ]:
por_cargo = saida.groupby('DS_CARGO', as_index=False)['patrimonio_total'].median().sort_values('patrimonio_total', ascending=False)
px.bar(por_cargo, x='DS_CARGO', y='patrimonio_total', title='Patrimônio mediano por cargo').show()
px.box(saida, x='SG_PARTIDO', y='patrimonio_total', title='Distribuição do patrimônio por partido').show()
print('As medianas permitem comparar grupos sem que poucos patrimônios extremos dominem a leitura.')

## A6. Narrativa

**Contexto:** os candidatos declaram seus bens à Justiça Eleitoral e esses dados permitem acompanhar a transparência patrimonial. **Insight:** após agregar os bens e cruzar com cargo e partido, a mediana revela diferenças entre grupos sem ser dominada por poucos valores extremos. **Conclusão:** o patrimônio declarado não é homogêneo; a leitura conjunta de patrimônio, cargo e partido mostra onde a concentração patrimonial está e torna a análise mais informativa que uma lista de bens isolados.